# **<font color='#6edce9'>Validacion de consolidado</font>**
### **Objetivo:** Construir el pipeline de mapeo y limpieza para unificar los datasets anuales de RRHH y migrar la base de datos consolidada a PostgreSQL.

## **<font color='#eda985'> 1. Importación y llamada a función para consolidar los 7 excel (2019 hasta 2025)</font>**

In [ ]:
import sys
from pathlib import Path
import pandas as pd

sys.path.append(str(Path.cwd().parent))  # sube un nivel desde notebooks/ hacia la raíz

from src.minsa_analystics.etl import consolidar_datos

# ============================================================
# CONFIGURACIÓN
# ============================================================
# --- Carga con caché en Parquet, para no re-procesar los 7 Excel cada vez ---
FORZAR_RECARGA = False # IMPORTANTE⚠️: . cambia a True cuando se modifique algo  etl.py o corrijas datos

ruta_datos = str(Path.cwd().parent / 'data' / 'raw')
ruta_parquet = Path.cwd().parent / 'data' / 'processed' / 'consolidado.parquet'

# ============================================================
# CARGA / GENERACIÓN DEL CONSOLIDADO
# ============================================================
if ruta_parquet.exists() and not FORZAR_RECARGA:
    print("Cargando consolidado desde cache (parquet)...")
    df = pd.read_parquet(ruta_parquet)
else:
    print("Generando consolidado desde los 7 Excel...")
    df = consolidar_datos(str(ruta_datos))

    # Crear carpeta si no existe
    ruta_parquet.parent.mkdir(parents=True, exist_ok=True)

    # Guardar Parquet
    df.to_parquet(ruta_parquet, index=False)

    print(f"\nParquet guardado en:")
    print(ruta_parquet)
    print(f"\n¿Archivo creado?: {ruta_parquet.exists()}")

    if ruta_parquet.exists():
        print(f"Tamaño: {ruta_parquet.stat().st_size / 1024**2:.2f} MB")


print(f"\nDimensiones: {df.shape}")

In [ ]:
# Verificando datos del campo 'es_especialista'
print(df.groupby(['anio_registro', 'es_especialista'], dropna=False).size().unstack(fill_value=0))


## **<font color='#eda985'> 2. Verificando cantidad de registros</font>**

In [ ]:
import pandas as pd
pd.set_option('display.float_format', '{:.4f}'.format)

# Filas por año (deberían coincidir con lo que ya viste arriba)
print('Cantidad de registros por año')
print('='*30)
print(df['anio_registro'].value_counts().sort_index())

In [ ]:
df.head(5)

In [ ]:
df.columns

In [ ]:
# Revisando tipo de dato de cada columna

# for col in df.columns:
#     tipos = df[col].map(type).unique()
#     if len(tipos) > 1:
#         print(col, '->', tipos)

for col in df.columns:
    tipos = df[col].map(type).unique()
    print(f'{col:<28} -> {tipos}')

## **<font color='#eda985'> 3. Verificando nulos de cada columna y filas con espacios al inicio o final</font>**

In [ ]:
# % de nulos por columna, para detectar algo inesperado
print('% Columnas Nulas')
print('='*30)
print(df.isna().mean().sort_values(ascending=False).head(10))

# Resultado inicial primer parquet
''' 
% Columnas Nulas
==============================
id_especialidad          0.7159
especialidad             0.7159
condicion_especialidad   0.7156
microrred                0.1153
red                      0.0835
edad                     0.0021
sexo                     0.0003
clasificacion            0.0003
distrito                 0.0000
provincia                0.0000
dtype: float64
'''

In [ ]:
# Verificando filas con espacios al inicio o al final
cols_texto = [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]
for col in cols_texto:
    s = df[col].dropna()
    n = (s != s.str.strip()).sum()
    if n:
        print(f'{col}: {n} filas con espacios al inicio o al final')

# Resultado inicial primer parquet
'''
departamento: 7 filas con espacios al inicio o al final
provincia: 7 filas con espacios al inicio o al final
distrito: 52 filas con espacios al inicio o al final
clasificacion: 3008 filas con espacios al inicio o al final
descripcionestablecimiento: 3895 filas con espacios al inicio o al final
categoria: 42 filas con espacios al inicio o al final
condicion_laboral: 311657 filas con espacios al inicio o al final
'''

In [ ]:
# Verificando filas con dobles espacios

cols = [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]
for col in cols:
    n = df[col].str.contains(r'\s{2,}', na=False).sum()
    if n:
        antes = df[col].nunique()
        despues = df[col].str.replace(r'\s+', ' ', regex=True).nunique()
        print(f'{col:<27}: {n} filas con dobles espacios | únicos {antes} -> {despues}')


# Resultado inicial primer parquet
''' 
uedescripue                : 9239 filas con dobles espacios | únicos 238 -> 238
red                        : 1058 filas con dobles espacios | únicos 161 -> 161
microrred                  : 325 filas con dobles espacios | únicos 882 -> 882
clasificacion              : 201 filas con dobles espacios | únicos 55 -> 55
descripcionestablecimiento : 63871 filas con dobles espacios | únicos 9802 -> 9780
especialidad               : 2579 filas con dobles espacios | únicos 356 -> 353
'''

In [ ]:
# Verificando registros con tildes
patron = r'[ÁÉÍÓÚÜáéíóúü]'
for col in [c for c in df.columns if pd.api.types.is_string_dtype(df[c])]:
    n = df[col].str.contains(patron, na=False).sum()
    if n:
        print(f'{col:<15}: {n} filas con tildes')


# Resultado inicial primer parquet
''' 
uedescripue: 6871 filas con tildes
provincia: 52 filas con tildes
distrito: 84 filas con tildes
clasificacion: 257 filas con tildes
descripcionestablecimiento: 26981 filas con tildes
categoria: 9967 filas con tildes
id_especialidad: 1 filas con tildes
especialidad: 116879 filas con tildes
regimen_laboral: 131867 filas con tildes
condicion_laboral: 131823 filas con tildes
grupofinal3: 427669 filas con tildes
'''

In [ ]:
# compara cuántos valores distintos tiene cada columna antes y después de limpiar las tildes
TILDES = str.maketrans('ÁÉÍÓÚÜáéíóúü', 'AEIOUUaeiouu')
cols = ['uedescripue', 'provincia', 'distrito', 'clasificacion', 'descripcionestablecimiento',
        'categoria', 'id_especialidad', 'especialidad', 'regimen_laboral',
        'condicion_laboral', 'grupofinal3']

for col in cols:
    antes = df[col].nunique()
    despues = df[col].str.strip().str.translate(TILDES).nunique()
    print(f'{col:<27}: {antes} -> {despues}')

    #print(f'{col:<15}: {n} filas con tildes')

# Resultado inicial primer parquet
''' 
uedescripue: 240 -> 240
provincia: 198 -> 196
distrito: 1737 -> 1734
clasificacion: 68 -> 55
descripcionestablecimiento: 9820 -> 9802
categoria: 27 -> 25
id_especialidad: 395 -> 395
especialidad: 495 -> 356
regimen_laboral: 12 -> 11
condicion_laboral: 72 -> 58
grupofinal3: 21 -> 21
'''

In [ ]:
# Distribución de las tildes en las columnas grandes
patron = r'[ÁÉÍÓÚÜáéíóúü]'
col = 'grupofinal3'
con_tilde = df[col].str.contains(patron, na=False)
print(df.loc[con_tilde].groupby('anio_registro').size())
print(df.loc[con_tilde, col].value_counts().head(10))

# Resultado inicial primer parquet
''' 
anio_registro
2019    44618
2020    55690
2021    69674
2022    61329
2023    71310
2024    61522
2025    63526
dtype: int64
grupofinal3
Médico                     257387
Odontólogo                  41847
Psicólogo                   38715
Tecnólogo Médico            31916
Químico Farmacéutico        28383
Biólogo                     24436
Médico Veterinario           2542
Técnicos Especializados      1901
Químico                       542
Name: count, dtype: int64
'''

## **<font color='#eda985'> 4. Verificando cada columna</font>**

In [ ]:
# ======== VERIFICACION CAMPO POR CAMPO ==============
def revisar(df, col, n=15):
    print(f'Tipo de dato: {df[col].dtype}')
    print('======= NULOS POR AÑO =======')
    print(df.groupby('anio_registro')[col].apply(lambda s: s.isna().sum()).to_frame('nulos'))
    print('======= ÚNICOS (TOTAL) =======')
    print(df[col].nunique(dropna=False))
    print('======= ÚNICOS POR AÑO =======')
    print(df.groupby('anio_registro')[col].nunique(dropna=False).to_frame('unicos'))
    vc = df[col].value_counts(dropna=False)
    print(f'======= {n} MÁS FRECUENTES =======')
    print(vc.head(n))
    print(f'======= {n} MENOS FRECUENTES =======')
    print(vc.tail(n))
    print('======= TIPOS MEZCLADOS POR AÑO =======')
    print(df.groupby('anio_registro')[col].agg(lambda s: s.map(type).nunique()))


### **<font color='#96f499'> 4.1. Verificando columna 'pea'  </font>**

In [ ]:
revisar(df, 'pea') # Constante = 1 

### **<font color='#96f499'> 4.2. Verificando columna 'renaes'  </font>**

In [ ]:
revisar(df, 'renaes')

### **<font color='#96f499'> 4.3. Verificando columna 'pliegodescrip'  </font>**

In [ ]:
revisar(df, 'pliegodescrip')

In [ ]:
col = 'pliegodescrip' 
df[col].value_counts(dropna=False).sort_values()

### **<font color='#96f499'> 4.4. Verificando columna 'uedescripue'  </font>**

In [ ]:
revisar(df, 'uedescripue')

In [ ]:
col = 'uedescripue' 
df[col].value_counts(dropna=False).sort_values()

In [ ]:
# ==== VERIFICANDO NOMBRES POR CODIGO =====

cod = df['uedescripue'].str.extract(r'^(\d+)')[0] # dividimos el texto extrayendo los digitos del texto
print(f'CODIGOS UNICOS DE UE:{cod.nunique()}')

print('\nCODIGOS CON MAS DE UNA DESCRIPCION')
nombres_por_codigo = df.assign(cod=cod).groupby('cod')['uedescripue'].nunique()
print(nombres_por_codigo[nombres_por_codigo > 1])   # códigos con más de un nombre

In [ ]:
# Como se ve que los dos codigo '0901' y '1712' tienen 2 descripciones diferetes evaluamos en que excel
for c in ['0901', '1712']:
    print(f'\n=== Código {c} ===')
    print(
        df.loc[cod == c]
        .groupby(['uedescripue', 'anio_registro']).size()
        .unstack(fill_value=0)
    )

### **<font color='#96f499'> 4.5. Verificando columna 'ubigeo'  </font>**

In [ ]:
revisar(df, 'ubigeo')

In [ ]:
# ========== VERIFICANDO LONGITUD DE DIGITOS = 6 ===========
print(df['ubigeo'].str.len().value_counts())          
print((~df['ubigeo'].str.fullmatch(r'\d{6}')).sum())  # debe dar 0

### **<font color='#96f499'> 4.6. Verificando columna 'departamento'  </font>**

In [ ]:
revisar(df, 'departamento')

In [ ]:
col = 'departamento' 
df[col].value_counts(dropna=False).sort_values()

In [ ]:
cruce = df.assign(cod_dep=df['ubigeo'].str[:2]).groupby('cod_dep')['departamento'].nunique()
print(cruce[cruce > 1])   # códigos de departamento con más de un nombre

In [ ]:
cod_dep = df['ubigeo'].str[:2]
print(
    df.loc[cod_dep == '13'].groupby('departamento').size()
)

In [ ]:
cod_dep = df['ubigeo'].str[:2]
print(
    df.loc[cod_dep == '13']
    .groupby(['departamento', 'anio_registro']).size()
    .unstack(fill_value=0)
)

cod_dep = df['ubigeo'].str[:2]

In [ ]:
print(df.loc[cod_dep == '13', 'departamento'].map(repr).value_counts())

### **<font color='#96f499'> 4.7. Verificando columna 'provincia'  </font>**

In [ ]:
revisar(df, 'provincia')

In [ ]:
cruce = df.assign(cod_prov=df['ubigeo'].str[:4]).groupby('cod_prov')['provincia'].nunique()
print(cruce[cruce > 1])   # códigos de provincia con más de un nombre

In [ ]:
cod_prov = df['ubigeo'].str[:4]
print(
    df.loc[cod_prov == '0809'].groupby('provincia').size()
)


# Resultado inicial primer parquet
''' 
provincia
LA CONVENCION    12992
LA CONVENCIÓN       52
dtype: int64
'''

In [ ]:
cod_prov = df['ubigeo'].str[:4]
print(
    df.loc[cod_prov == '0809'].groupby(['provincia', 'anio_registro']).size().unstack(fill_value=0)
)

cod_prov = df['ubigeo'].str[:4]

# Resultado inicial primer parquet
'''
anio_registro  2019  2020  2021  2022  2023  2024  2025
provincia                                              
LA CONVENCION  1480  1637  1854  1899  1951  1996  2175
LA CONVENCIÓN     0     0     0     0     0    52     0 
'''


### **<font color='#96f499'> 4.8. Verificando columna 'distrito'  </font>**

In [ ]:
revisar(df, 'distrito')

In [ ]:
# Un solo nombre de distrito por cada ubigeo (6 dígitos)
cruce = df.groupby('ubigeo')['distrito'].nunique()
print(cruce[cruce > 1])

# Resultado inicial primer parquet
''' 
ubigeo
130112    2
Name: distrito, dtype: int64
'''


In [ ]:
cod_distrito = df['ubigeo']
print(
    df.loc[cod_distrito == '130112'].groupby('distrito').size()
)


# Resultado inicial primer parquet
''' 
distrito
ALTO TRUJILLO    130
VIR                7
dtype: int64
'''

In [ ]:
cod_distrito = df['ubigeo']
print(
    df.loc[cod_distrito == '130112'].groupby(['distrito', 'anio_registro']).size().unstack(fill_value=0)
)



# Resultado inicial primer parquet
'''
anio_registro  2024  2025
distrito                 
ALTO TRUJILLO     0   130
VIR               7     0 
'''


In [ ]:
print(df.loc[(df['ubigeo'] == '130112') & (df['distrito'] == 'VIR'),
             ['anio_registro', 'provincia', 'renaes', 'descripcionestablecimiento']])

# Resultado inicial primer parquet
''' 
anio_registro provincia    renaes descripcionestablecimiento
1514016           2024  TRUJILLO  00012229                    BARRIO1
1629240           2024  TRUJILLO  00005220             ALTO  TRUJILLO
1632038           2024  TRUJILLO  00005220             ALTO  TRUJILLO
1632049           2024  TRUJILLO  00005220             ALTO  TRUJILLO
1632067           2024  TRUJILLO  00005220             ALTO  TRUJILLO
1632233           2024  TRUJILLO  00012229                    BARRIO1
1632366           2024  TRUJILLO  00005220             ALTO  TRUJILLO
'''

In [ ]:
# COMPROBACION UBICACION GEOGRAFICA (departamento,, provincia, distrito)
chequeo = df.groupby('ubigeo')[['departamento', 'provincia', 'distrito']].nunique()
print(chequeo[(chequeo > 1).any(axis=1)])

# Resultado inicial primer parquet
''' 
        departamento  provincia  distrito
ubigeo                                   
130112             1          1         2
'''

### **<font color='#96f499'> 4.9. Verificando columna 'diresa'  </font>**

In [ ]:
revisar(df, 'diresa')

In [ ]:
col = 'diresa' 
df[col].value_counts(dropna=False).sort_values()

### **<font color='#96f499'> 4.10. Verificando columna 'red'  </font>**

In [ ]:
revisar(df, 'red')

In [ ]:
col = 'red' 
df[col].value_counts(dropna=False).sort_values()


In [ ]:
col = "red"
s = df[col]
contiene = s.str.contains("a ninguna", case=False, na=False)

print(f"'-' total: {(s == '-').sum():,}")
print(f"NaN total: {s.isna().sum():,}")
print(f"Contienen 'a ninguna': {contiene.sum():,}")
print(s[contiene].value_counts())   # cada variante con su conteo


### **<font color='#96f499'> 4.11. Verificando columna 'microrred'  </font>**

In [ ]:
revisar(df, 'microrred')

In [ ]:
col = 'microrred' 
df[col].value_counts(dropna=False).sort_values()

In [ ]:
col = "microrred"
s = df[col]
contiene = s.str.contains("a ninguna", case=False, na=False)

print(f"'-' total: {(s == '-').sum():,}")
print(f"NaN total: {s.isna().sum():,}")
print(f"Contienen 'a ninguna': {contiene.sum():,}")
print(s[contiene].value_counts())   # cada variante con su conteo

### **<font color='#96f499'> 4.12. Verificando columna 'clasificacion'  </font>**

In [ ]:
revisar(df, 'clasificacion')

In [ ]:
col = 'clasificacion' 
df[col].value_counts(dropna=False).sort_values()

### **<font color='#96f499'> 4.1. Verificando columna 'descripcionestablecimiento'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'categoria'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'quintil'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'pcm'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'distfrontera'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'zaf2014final'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'estrategicos'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'sexo'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'es_especialista'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'id_especialidad'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'especialidad'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'condicion_especialidad'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'regimen_laboral'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'condicion_laboral'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'id_cargo'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'cargo'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'grupofinal2'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'grupofinal3'  </font>**

### **<font color='#96f499'> 4.1. Verificando columna 'edad'  </font>**

## **<font color='#eda985'> 4. Auditoría y Perfilado de Errores</font>**

### **<font color='#96f499'> 4.1. Verificando y corrigiendo campos Relacionados: id_especialidad, especialidad,condicion_especialidad, es_especialista </font>**

### Reglas de Negocio: Validación de Especialistas
Para validar la consistencia de los datos, la columna `es_especialista` debe cruzarse obligatoriamente con los campos de especialidad (`id_especialidad`, `especialidad`, `condicion_especialidad`) y `la condición laboral`.

##### 1. Cuando `es_especialista` es **"SI"** 🟢
**Regla estricta:** Todos los campos de especialidad deben estar **completamente llenos**.
**Campos obligatorios:**
  * `id_especialidad`
  * `especialidad`
  * `condicion_especialidad`
*Cualquier registro en "SI" con alguno de estos campos vacío se considera un **Error de Consistencia**.*

##### 2. Cuando `es_especialista` es **"NO"** 🟡
* **Regla flexible:** Los campos de especialidad **pueden o no estar llenos** (se permite que tengan datos o que estén vacíos).
* **Excepción por Residencia:** Si el registro tiene `condicion_laboral` = **"residente"**, el valor de `es_especialista` será obligatoriamente **"NO"**, incluso si ya tiene asignada una especialidad en los campos de texto.

---

> 💡 **Nota de Diagnóstico:** Tras analizar el set de datos, los **11 registros** con valor "SI" y especialidad vacía (`False`) representan la inconsistencia más grave, ya que violan la regla estricta del negocio. Los registros en "NO", "No" o "0" que contienen datos pueden justificarse bajo la excepción de médicos residentes.
Usa el código con precaución.
🛠️ Código de Pandas actualizado con la nueva regla
Para que tu script refleje exactamente esta nueva consideración de los residentes, puedes usar este enfoque para aislar únicamente los errores reales (los 11 "SI" vacíos y cualquier "SI" que sea residente por error):
python

In [ ]:
# 1. Filtro estricto: Es "SI" pero le falta alguna de las 3 columnas de especialidad
campos_incompletos = df['id_especialidad'].isna() | df['especialidad'].isna() | df['condicion_especialidad'].isna()
error_si_incompleto = (df['es_especialista'] == 'SI') & campos_incompletos

# 2. Filtro de contradicción: Dice "SI" pero su condición laboral es "residente" (debería ser NO)
error_residente_especialista = (df['es_especialista'] == 'SI') & (df['condicion_laboral'].str.lower() == 'residente')

# Unimos los errores reales del sistema
errores_reales = df[error_si_incompleto | error_residente_especialista]

print(f"Total de registros con errores críticos detectados: {len(errores_reales)}")
print(f'Total regitros indican SI es especialista pero alguno de los 3 campos vacios: {error_si_incompleto.sum()}')
print(f'Total regitros indican SI es especialista pero condicion laboral es Residente: {error_residente_especialista.sum()} ')

In [ ]:
# Verificando campo 'tiene_especialidad' con 'especialidad' NO VACÍO
df['tiene_especialidad'] = df['especialidad'].notna() # es_especialista vs especialidad llena
print(df.groupby(['es_especialista', 'tiene_especialidad'], dropna=False).size())

### **<font color='#96f499'> 4.2. Verificando detalle del campo EDAD </font>**

In [ ]:
# EDAD: rango razonable en todo el consolidado
print('Descripcion EDAD')
print('='*30)
print(df['edad'].describe())

In [ ]:
# Registros con edad mayo a 80 años
df[df['edad'] > 80][['edad', 'anio_registro']].value_counts()